# 节点 05 — 梯度消失实验

**目标**：用 numpy 从零实现一个简单 RNN，
亲眼看到梯度随时间步数的衰减。

**前置知识**：会 Python 列表和 for 循环；知道梯度是「误差的方向」。

**运行要求**：只需 numpy，无需 GPU。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 固定随机种子，保证结果可复现
np.random.seed(42)
print('numpy version:', np.__version__)

## Part 1：传话游戏模拟

先用最简单的方式展示指数衰减。

每次传话，信号乘以一个系数 δ（delta）。
δ < 1 → 消失；δ > 1 → 爆炸；δ = 1 → 稳定。

In [ ]:
def simulate_signal_propagation(delta, n_steps):
    """模拟信号经过 n_steps 次传递后的大小（从1.0开始）。"""
    signal = 1.0
    history = [signal]
    for _ in range(n_steps):
        signal = signal * delta
        history.append(signal)
    return history

steps = 30
deltas = [0.9, 0.8, 0.5, 1.0]
labels = ['δ=0.9（慢消失）', 'δ=0.8', 'δ=0.5（快消失）', 'δ=1.0（稳定）']

plt.figure(figsize=(10, 4))
for delta, label in zip(deltas, labels):
    history = simulate_signal_propagation(delta, steps)
    plt.plot(history, label=label, linewidth=2)

plt.xlabel('传递步数（Time Steps）')
plt.ylabel('信号大小（梯度大小）')
plt.title('传话游戏：δ < 1 时信号指数消失')
plt.legend()
plt.yscale('log')  # 对数坐标，让衰减看得更清楚
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('signal_decay.png', dpi=100)
plt.show()
print('图已保存: signal_decay.png')

## Part 2：Sigmoid 函数及其导数

Sigmoid 导数的最大值是 0.25——这是梯度消失的根本原因。

In [ ]:
def sigmoid(x):
    """Sigmoid 激活函数：把任意实数压缩到 (0, 1)。"""
    return 1.0 / (1.0 + np.exp(-x))

def sigmoid_derivative(x):
    """Sigmoid 的导数：σ(x) * (1 - σ(x))，最大值 = 0.25。"""
    s = sigmoid(x)
    return s * (1.0 - s)

# 验证最大值
x_vals = np.linspace(-6, 6, 500)
deriv_vals = sigmoid_derivative(x_vals)
max_deriv = deriv_vals.max()
print(f'Sigmoid 导数最大值 = {max_deriv:.4f}（理论值 = 0.25）')
assert abs(max_deriv - 0.25) < 0.001, '最大值应接近 0.25'

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(x_vals, sigmoid(x_vals), color='steelblue', linewidth=2)
axes[0].set_title('Sigmoid 函数 σ(x)')
axes[0].set_xlabel('x')
axes[0].set_ylabel('σ(x)')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='y=0.5')
axes[0].legend()

axes[1].plot(x_vals, deriv_vals, color='tomato', linewidth=2)
axes[1].axhline(0.25, color='gray', linestyle='--', alpha=0.8, label=f'最大值 = 0.25')
axes[1].set_title("Sigmoid 导数 σ'(x) — 最大值仅 0.25")
axes[1].set_xlabel('x')
axes[1].set_ylabel("σ'(x)")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.savefig('sigmoid_and_derivative.png', dpi=100)
plt.show()
print('图已保存: sigmoid_and_derivative.png')

## Part 3：从零实现简单 RNN

RNN 每步：`h_t = sigmoid(W_h * h_{t-1} + W_x * x_t + b)`

反向传播时，梯度要通过每一步的 `sigmoid'` 和 `W_h` 传回去。

In [ ]:
class SimpleRNN:
    """
    单个隐藏单元的最简 RNN（用于演示梯度消失）。

    每步计算：h_t = sigmoid(W_h * h_{t-1} + W_x * x_t + b)
    输出：y_T = h_T（最后一步隐藏状态作为预测）

    为了清晰，这里只用标量（1维），不用矩阵。
    """

    def __init__(self, W_h=0.9, W_x=0.5, b=0.0):
        self.W_h = W_h   # 隐藏状态权重（循环权重）
        self.W_x = W_x   # 输入权重
        self.b = b       # 偏置

    def forward(self, inputs):
        """
        前向传播：按序列处理每个输入。
        inputs: list of scalars，长度 = T（时间步数）
        返回：隐藏状态列表 [h_0, h_1, ..., h_T]
        """
        h = 0.0  # 初始隐藏状态
        hidden_states = [h]
        for x_t in inputs:
            z = self.W_h * h + self.W_x * x_t + self.b
            h = sigmoid(z)
            hidden_states.append(h)
        return hidden_states

    def compute_gradient_at_step(self, hidden_states, target_step):
        """
        计算损失对第 target_step 步之前梯度的大小。

        直觉：误差从最后一步往回传，每过一步，
        梯度就要乘以 sigmoid'(z_t) * W_h。

        返回：从最后一步传到每个历史步骤的梯度模（列表）
        """
        T = len(hidden_states) - 1  # 总步数
        gradient_magnitudes = []

        # 从最后一步往回累积梯度乘积
        grad = 1.0  # dL/dh_T = 1（简化损失）
        for t in range(T, 0, -1):
            h_t = hidden_states[t]
            # 链式法则：dh_t/dh_{t-1} = sigmoid'(z_t) * W_h
            # 用 h_t 反推 sigmoid'：σ' = h_t * (1 - h_t)
            local_grad = h_t * (1.0 - h_t) * self.W_h
            grad *= local_grad
            gradient_magnitudes.append(abs(grad))

        # 反转：让索引 0 = 距离 1 步，索引 T-1 = 距离 T 步
        return gradient_magnitudes


# 测试：10 步序列
rnn = SimpleRNN(W_h=0.9, W_x=0.5, b=0.0)
test_inputs = [0.5] * 10
hidden = rnn.forward(test_inputs)
grads = rnn.compute_gradient_at_step(hidden, target_step=0)
print(f'前向传播完成，隐藏状态数量: {len(hidden)}')
print(f'梯度传播到前 10 步的大小: {[f"{g:.4f}" for g in grads]}')

## Part 4：可视化梯度消失

实验：同样的 RNN，序列越长，最早那步的梯度越小。

In [ ]:
def measure_gradient_decay(rnn, sequence_length):
    """
    给定序列长度，返回从最后一步往前每步的梯度大小。
    输入序列全为 0.5（任意常数，影响不大）。
    """
    inputs = [0.5] * sequence_length
    hidden_states = rnn.forward(inputs)
    return rnn.compute_gradient_at_step(hidden_states, target_step=0)

# 三种不同权重的 RNN
rnns = [
    (SimpleRNN(W_h=0.9), 'W_h=0.9（权重接近 1）'),
    (SimpleRNN(W_h=0.5), 'W_h=0.5（权重中等）'),
    (SimpleRNN(W_h=0.3), 'W_h=0.3（权重小）'),
]

seq_len = 50
plt.figure(figsize=(12, 5))

for rnn_obj, label in rnns:
    grads = measure_gradient_decay(rnn_obj, seq_len)
    steps_back = list(range(1, len(grads) + 1))
    plt.plot(steps_back, grads, label=label, linewidth=2)

plt.xlabel('往前回溯的步数（Steps Back）')
plt.ylabel('梯度大小（Gradient Magnitude）')
plt.title('梯度消失：步数越多，梯度越小')
plt.yscale('log')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axhline(1e-6, color='red', linestyle='--', alpha=0.5, label='实际可用阈值')
plt.tight_layout()
plt.savefig('gradient_vanishing.png', dpi=100)
plt.show()
print('图已保存: gradient_vanishing.png')

# 数字验证：50步后梯度应接近0
final_grad_weak = measure_gradient_decay(SimpleRNN(W_h=0.5), 50)[-1]
print(f'\nW_h=0.5 经过 50 步后梯度大小: {final_grad_weak:.2e}')
assert final_grad_weak < 1e-5, f'梯度应该接近 0，实际: {final_grad_weak}'

## Part 5：梯度爆炸也可能发生

如果权重 W_h > 1，梯度不消失，反而**指数爆炸**。
两种极端都无法训练——这就是 Bengio 1994 的核心结论。

In [ ]:
def linear_gradient(W_h_val, seq_len):
    """
    线性 RNN（无激活函数）的梯度传播公式：梯度 = W_h^seq_len。
    用线性模型清楚展示消失/爆炸的临界点，因为 sigmoid 会限制上界。
    """
    return abs(W_h_val) ** seq_len

W_h_values = [0.3, 0.5, 0.8, 0.9, 1.0, 1.1, 1.5, 2.0]
seq_len = 20
print('线性 RNN 梯度（W_h^20）：')
for w in W_h_values:
    g = linear_gradient(w, seq_len)
    trend = '消失' if g < 1e-3 else ('爆炸' if g > 1e3 else '接近稳定')
    print(f'  W_h={w:.1f}: 梯度={g:.2e} [{trend}]')

import numpy as np
import matplotlib.pyplot as plt
W_range = np.linspace(0.1, 2.5, 100)
grads_20 = [linear_gradient(w, 20) for w in W_range]

plt.figure(figsize=(10, 4))
plt.plot(W_range, grads_20, 'steelblue', linewidth=2)
plt.axvline(1.0, color='red', linestyle='--', label='W_h = 1（临界点）')
plt.axhline(1.0, color='gray', linestyle=':', alpha=0.5)
plt.xlabel('循环权重 W_h')
plt.ylabel('20 步后梯度大小（对数坐标）')
plt.title('线性 RNN：W_h < 1 消失；W_h > 1 爆炸')
plt.yscale('log')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('vanishing_vs_exploding.png', dpi=100)
plt.show()
print('图已保存: vanishing_vs_exploding.png')

# 验证：W_h<1 消失，W_h>1 爆炸（线性模型，没有 sigmoid 的限制）
grad_small = linear_gradient(0.5, 20)
grad_large = linear_gradient(1.5, 20)
print(f'W_h=0.5（消失区域）：梯度 = {grad_small:.2e}')
print(f'W_h=1.5（爆炸区域）：梯度 = {grad_large:.2e}')
assert grad_small < 1e-3, '小权重应该导致梯度消失'
assert grad_large > 1e3, '大权重应该导致梯度爆炸'
print('验证通过：线性 RNN 清楚展示了消失/爆炸的临界点')

## Part 6：门控的直觉（LSTM 的核心思想预览）

LSTM 的解决方案：用一个**门（gate）**控制信息是否通过。

门值 = 1：信息完整通过，梯度不缩小。  
门值 = 0：信息完全阻断。

这样，梯度不需要每步都乘以 sigmoid'——它可以沿「记忆通道」直接回传。

In [ ]:
def gated_rnn_gradient(gate_val, seq_len):
    """
    模拟有门控的梯度传播：每步梯度乘以 gate（固定值，简化演示）。
    gate=1 → 梯度不消失；gate<1 → 部分消失。
    """
    grad = 1.0
    history = []
    for _ in range(seq_len):
        grad *= gate_val
        history.append(abs(grad))
    return history

seq_len = 50
plt.figure(figsize=(10, 4))

# 普通 RNN：梯度乘以 sigmoid' * W_h ≈ 0.25 * 0.9 ≈ 0.225
vanilla_grads = gated_rnn_gradient(0.225, seq_len)
plt.plot(vanilla_grads, label='普通 RNN（每步 ×0.225）', color='tomato', linewidth=2)

# LSTM 理想情况：门=1，梯度不消失
lstm_grads = gated_rnn_gradient(1.0, seq_len)
plt.plot(lstm_grads, label='LSTM 门=1（梯度直接通过）', color='green', linewidth=2)

# LSTM 折中：门=0.95
partial_grads = gated_rnn_gradient(0.95, seq_len)
plt.plot(partial_grads, label='LSTM 门=0.95（缓慢衰减）', color='steelblue', linewidth=2)

plt.xlabel('往前回溯的步数')
plt.ylabel('梯度大小')
plt.title('门控机制对比：LSTM 让梯度活得更久')
plt.yscale('log')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(1e-15, 10)
plt.tight_layout()
plt.savefig('gated_vs_vanilla.png', dpi=100)
plt.show()
print('图已保存: gated_vs_vanilla.png')

# 验证：门=1时梯度保持为1
assert abs(lstm_grads[-1] - 1.0) < 1e-10, 'gate=1 时梯度应保持不变'
print('验证通过：gate=1 时 50 步后梯度仍为', lstm_grads[-1])

## 总结

| 现象 | 根本原因 |
|------|----------|
| 梯度消失 | sigmoid' ≤ 0.25，每步乘一次，T步后 → 0 |
| 梯度爆炸 | 权重大 → 每步乘数 > 1，T步后 → ∞ |
| LSTM 解法 | 门控让梯度「绕过」sigmoid，直接回传 |

**关键数字**：Sigmoid 导数最大值 **0.25**，这是整个问题的来源。

**历史意义**：Hochreiter 1991 年用这个分析，奠定了 LSTM、ReLU、ResNet 的理论基础。

In [ ]:
# 最终验证：所有核心不等式
print('=== 最终验证 ===')

# 1. sigmoid 导数最大值
x_test = np.linspace(-10, 10, 10000)
max_sd = sigmoid_derivative(x_test).max()
assert abs(max_sd - 0.25) < 0.001, f'sigmoid 导数最大值应为 0.25，得到 {max_sd}'
print(f'✓ sigmoid 导数最大值 = {max_sd:.4f}（≈ 0.25）')

# 2. 消失验证：0.25^20 < 1e-12
vanish_20 = 0.25 ** 20
assert vanish_20 < 1e-10, f'0.25^20 应极小，实际 {vanish_20}'
print(f'✓ 0.25^20 = {vanish_20:.2e}（约等于零）')

# 3. RNN 梯度验证
rnn_test = SimpleRNN(W_h=0.5)
grads_test = measure_gradient_decay(rnn_test, 30)
assert grads_test[-1] < 1e-5, f'30步后梯度应接近0，实际 {grads_test[-1]}'
print(f'✓ SimpleRNN (W_h=0.5) 30步后梯度 = {grads_test[-1]:.2e}')

# 4. gate=1 梯度不变
gate1_grads = gated_rnn_gradient(1.0, seq_len=50)
assert abs(gate1_grads[-1] - 1.0) < 1e-10
print(f'✓ gate=1 时 50步后梯度 = {gate1_grads[-1]:.4f}（不消失）')

print('\n所有验证通过！')